In [0]:
%sql
SHOW CATALOGS

In [0]:
%sql
USE CATALOG uci_catalog

In [0]:
%pip install openpyxl

In [0]:
 %restart_python

In [0]:
import pandas as pd

excel_path = "/Volumes/uci_catalog/uci_schema/uci_volume/online_retail_II.xlsx"
csv_path = "/Volumes/uci_catalog/uci_schema/uci_volume/online_retail_II.csv"

# Read Excel
df = pd.read_excel(excel_path, engine="openpyxl")

# Convert to CSV
df.to_csv(csv_path, index=False, encoding="utf-8")

print(f"Conversion completed. Rows: {len(df)}")

In [0]:
%sql
select count(*) from 
read_files(
    '/Volumes/uci_catalog/uci_schema/uci_volume/online_retail_II.csv',
    format => 'csv'
)
LIMIT 10



In [0]:
df = spark.read.format("csv") \
              .option("header", "true") \
              .option("inferSchema", "true") \
              .load('/Volumes/uci_catalog/uci_schema/uci_volume/online_retail_II.csv')
display(df)

In [0]:
df.columns

In [0]:
df.printSchema()

In [0]:
display(df.limit(10))

In [0]:
from pyspark.sql import functions as F
#one invoice can have multiple transactions and it can contain repeated products 
df.groupBy("Invoice").agg(
      F.count("*").alias("transanction_lines"),
      F.countDistinct("StockCode").alias("unique_product_counts")
    ).orderBy(F.desc("transanction_lines")).show(10)

In [0]:
df.filter(df.Invoice == "537434").display()

In [0]:
df.groupBy("Invoice").agg(
    F.countDistinct("Customer ID").alias("unique_customer_count"),
    F.countDistinct("InvoiceDate").alias("unique_date_count")
).filter(
    (F.col("unique_customer_count") > 2) | (F.col("unique_date_count") > 1)
).show(10)

In [0]:

df.filter(F.col("Invoice") == '492807').display()

In [0]:
from pyspark.sql import functions as F

inconsistent_invoices = (
    df.groupBy("Invoice")
      .agg(F.countDistinct("InvoiceDate").alias("date_count"))
      .filter(F.col("date_count") > 1)
      .select("Invoice")
)

display(
    df.join(inconsistent_invoices, on="Invoice", how="inner")
      .orderBy("Invoice", "InvoiceDate")
      .limit(30)
)

In [0]:
df.filter(F.col("Invoice") == "492807") \
  .select("Invoice", "InvoiceDate", "Customer ID", "Country") \
  .distinct() \
  .show(truncate=False)

In [0]:
from pyspark.sql import functions as F

df.groupBy("StockCode").agg(
    F.countDistinct("Description").alias("description_count")
).filter(
    F.col("description_count") > 1
).orderBy(F.desc("description_count")).show(10, truncate=False)

In [0]:
df.filter(F.col("StockCode") == '22423').display()

In [0]:
df.filter(F.col("StockCode").isNull()  | F.trim(F.col("StockCode") == "")).count()

In [0]:
from pyspark.sql import functions as F

product_description_check = (
    df.groupBy("StockCode")
      .agg(
          F.countDistinct("Description").alias("description_count")
      )
)

product_description_check.select(
    F.count("*").alias("total_stock_codes"),
    F.sum(
        F.when(F.col("description_count") > 1, 1).otherwise(0)
    ).alias("codes_with_multiple_descriptions")
).show()

In [0]:
from pyspark.sql import functions as F

df.groupBy("Invoice").agg(
    F.countDistinct("Customer ID").alias("Unique_customer_count")
).filter(F.col("Unique_customer_count") > 1).display()

In [0]:
df.filter(F.col("Customer ID").isNotNull()).select("Customer ID").distinct().count()

In [0]:
df.select("Description").distinct().display()